# Day 35 — Fine-tune top layers at low LR

Status: COMPLETE. Unfroze `features[-3:]` + head (1.2M trainable, early
edge/texture filters stay locked), 3 epochs at 1e-4 (10x lower). 12/12 tests pass.

In [1]:
import json
from pathlib import Path

m = json.loads(Path("../models/finetune_metrics.json").read_text())
print(f"finetuned: accuracy {m['accuracy']} | ROC-AUC {m['roc_auc']} | "
      f"PR-AUC {m['pr_auc']}")
print(f"F1 {m['f1']} | precision {m['precision']}, recall {m['recall']} "
      "(only 4 missed pneumonias)")
print("vs frozen: 0.9584 -> 0.9622 ROC (+0.004), recall 0.972 -> 0.990,")
print("           precision 0.837 -> 0.832 (tiny cost). Model: 9.1 MB.")

finetuned: accuracy 0.8686 | ROC-AUC 0.9622 | PR-AUC 0.9714
F1 0.9040 | precision 0.8319, recall 0.9897 (only 4 missed pneumonias)
vs frozen: 0.9584 -> 0.9622 ROC (+0.004), recall 0.972 -> 0.990,
           precision 0.837 -> 0.832 (tiny cost). Model: 9.1 MB.


## Findings

1. **Small, honest gain.** +0.004 ROC, +0.018 recall at −0.005 precision: the
   top blocks adapted to X-ray textures without wrecking the backbone — the
   low LR did its job. No overfitting blowup (val would have shown it).
2. **Recall 0.99 is the triage number that matters**: 4 missed cases out of 390.
   Precision 0.83 means ~1 in 6 flags is a false alarm — acceptable for
   prioritized review, never for autonomous diagnosis.
3. **Diminishing returns are the signal to stop**: frozen got +0.10, fine-tuning
   +0.004. Further unfreezing on CPU buys hours for thousandths — Day 36 spends
   the budget on evaluation + explainability instead.